In [1]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 54.3 MB/s eta 0:00:00


In [3]:
!pip install seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=b35c0de735fc3fb7e5aac06428f22b26511bd5a92e93c8d029f07fa0e04ea3bb
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [4]:
!pip install tensorflow

In [5]:
import gdown
import os

os.makedirs("data", exist_ok=True)

# Link Google Drive
url = "https://drive.google.com/uc?id=1oMGJwujOeDIfNIGz9t8Uedm73ELN8qaN"

# Lokasi file output
output_path = "data/NER_dataset.csv"

print("Downloading dataset from Google Drive")
gdown.download(url, output_path, quiet=False)

print("Download complete!")
print("File saved to:", output_path)

import pandas as pd

df = pd.read_csv("data/NER_dataset.csv", encoding="latin1")

print("Dataset shape:", df.shape)
print(df.head())



Downloading...
From: https://drive.google.com/uc?id=1oMGJwujOeDIfNIGz9t8Uedm73ELN8qaN
To: /content/data/NER_dataset.csv
100%|██████████| 15.2M/15.2M [00:00<00:00, 122MB/s]


Download complete!
File saved to: data/NER_dataset.csv
Dataset shape: (1048575, 4)
    Sentence #           Word  POS Tag
0  Sentence: 1      Thousands  NNS   O
1          NaN             of   IN   O
2          NaN  demonstrators  NNS   O
3          NaN           have  VBP   O
4          NaN        marched  VBN   O


In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Checking the Structure of the Data

In [7]:
print(df.head())
print(df.isna().sum())
print(df['Tag'].value_counts())

    Sentence #           Word  POS Tag
0  Sentence: 1      Thousands  NNS   O
1          NaN             of   IN   O
2          NaN  demonstrators  NNS   O
3          NaN           have  VBP   O
4          NaN        marched  VBN   O
Sentence #    1000616
Word               10
POS                 0
Tag                 0
dtype: int64
Tag
O        887908
B-geo     37644
B-tim     20333
B-org     20143
I-per     17251
B-per     16990
I-org     16784
B-gpe     15870
I-geo      7414
I-tim      6528
B-art       402
B-eve       308
I-art       297
I-eve       253
B-nat       201
I-gpe       198
I-nat        51
Name: count, dtype: int64


## Filling the NAN values

In [8]:
df = df.ffill()

## Grouping words and tags by sentence

In [9]:
from collections import defaultdict

sentences=[]
labels=[]

grouped=df.groupby('Sentence #')

for _,group in grouped:
    words=list(group['Word'])
    tags=list(group['Tag'])
    sentences.append(words)
    labels.append(tags)

## Encode Labels (NER Tags)

In [10]:
from sklearn.preprocessing import LabelEncoder


le=LabelEncoder()

# Flatten all the tags into one big list
all_tags=list(set(tag for seq in labels for tag in seq))
le.fit(all_tags)

# Build two dictionaries to map back and forth
label2id = {tag: idx for idx, tag in enumerate(le.classes_)}
id2label = {idx: tag for tag, idx in label2id.items()}

## Tokenization & Label Alignment

In [11]:
from transformers import BertTokenizerFast

tokenizer = BertTokenizerFast.from_pretrained('bert-base-cased')

def tokenize_and_align_labels(sentences, labels):
    tokenized_inputs = tokenizer(
        sentences,
        is_split_into_words=True,
        padding=True,
        truncation=True,
        return_tensors="pt"
    )

    aligned_labels = []
    for i, label in enumerate(labels):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label2id[label[word_idx]])
            else:
                # You could use the same label or 'I-' version
                label_ids.append(label2id[label[word_idx]])
            previous_word_idx = word_idx
        aligned_labels.append(label_ids)

    tokenized_inputs["labels"] = aligned_labels
    return tokenized_inputs

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

## Creating a Dataset Class

In [12]:
from torch.utils.data import Dataset

class NERDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __getitem__(self, idx):
        return {key: val[idx] for key, val in self.encodings.items()}

    def __len__(self):
        return len(self.encodings["input_ids"])

## Train_test+_split

In [13]:
from sklearn.model_selection import train_test_split

train_sentences, val_sentences, train_labels, val_labels = train_test_split(
    sentences, labels, test_size=0.1, random_state=42
)

train_encodings = tokenize_and_align_labels(train_sentences, train_labels)
val_encodings = tokenize_and_align_labels(val_sentences, val_labels)

train_dataset = NERDataset(train_encodings)
val_dataset = NERDataset(val_encodings)

In [ ]:
print("Type of train_sentences:", type(train_sentences))
print("Contoh isi train_sentences:")
print(train_sentences[:2])

print("\nType of train_labels:", type(train_labels))
print("Contoh isi train_labels:")
print(train_labels[:2])


Type of train_sentences: <class 'list'>
Contoh isi train_sentences:
[['Kremlin', 'officials', 'insisted', 'that', 'they', 'are', 'cracking', 'down', 'on', 'corporate', 'crime', '.'], ['The', 'conflict', 'in', 'Sudan', "'s", 'western', 'Darfur', 'region', 'and', 'the', 'faltering', 'peace', 'process', 'in', 'Ivory', 'Coast', 'were', 'also', 'likely', 'to', 'top', 'the', 'agenda', '.']]

Type of train_labels: <class 'list'>
Contoh isi train_labels:
[['B-org', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'], ['O', 'O', 'O', 'B-geo', 'O', 'B-geo', 'I-geo', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-geo', 'I-geo', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']]


## Model

### 1. Code Base (From Template)

In [ ]:
# from transformers import BertForTokenClassification , Trainer , TrainingArguments ,DataCollatorForTokenClassification
# import os

# os.environ["WANDB_DISABLED"] = "true"

# data_collator = DataCollatorForTokenClassification(tokenizer)

# model=BertForTokenClassification.from_pretrained(
#     'bert-base-cased',
#     num_labels=len(label2id),
#     id2label=id2label,
#     label2id=label2id
# )

# training_args = TrainingArguments(
#     output_dir="./results",
#     learning_rate=2e-5,
#     per_device_train_batch_size=16,
#     per_device_eval_batch_size=16,
#     num_train_epochs=3,
#     weight_decay=0.01,
#     logging_steps=500,     # log every 500 steps
#     save_steps=1000,       # save every 1000 steps
#     save_total_limit=1     # keep only 1 checkpoint
# )

# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_dataset,
#     eval_dataset=val_dataset,
#     data_collator=data_collator # Removed tokenizer=tokenizer
# )

# trainer.train()

### 3. Teknik 3 : Transformer

#### Albert

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
import os
os.environ["WANDB_DISABLED"] = "true"

# Model
model_name = "albert-base-v2"

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

data_collator = DataCollatorForTokenClassification(tokenizer)

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="./results/albert",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=7,
    weight_decay=0.01,
    logging_steps=500,
    save_steps=1000,
    save_total_limit=1,
    fp16=True,
    optim="adamw_torch",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

trainer.train()

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/760k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.31M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/47.4M [00:00<?, ?B/s]

Some weights of AlbertForTokenClassification were not initialized from the model checkpoint at albert-base-v2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
500,0.936600
1000,0.716900
1500,0.575500
2000,0.495200
2500,0.411900
3000,0.361900
3500,0.330400
4000,0.307100
4500,0.293600
5000,0.264500


TrainOutput(global_step=37772, training_loss=0.16545039049892069, metrics={'train_runtime': 3431.6657, 'train_samples_per_second': 88.045, 'train_steps_per_second': 11.007, 'total_flos': 2036265832685448.0, 'train_loss': 0.16545039049892069, 'epoch': 7.0})

###### Evaluation

In [ ]:
from seqeval.metrics import classification_report

preds = trainer.predict(val_dataset)
predictions = preds.predictions.argmax(-1)
label_ids = preds.label_ids

true_labels = []
pred_labels = []

for pred, label in zip(predictions, label_ids):
    true_seq = []
    pred_seq = []
    for p, l in zip(pred, label):
        if l != -100:
            true_seq.append(id2label[l])
            pred_seq.append(id2label[p])
    true_labels.append(true_seq)
    pred_labels.append(pred_seq)

print(classification_report(true_labels, pred_labels))

              precision    recall  f1-score   support

         art       0.07      0.01      0.02        92
         eve       0.13      0.10      0.11        42
         geo       0.83      0.85      0.84      6167
         gpe       0.94      0.90      0.92      1790
         nat       0.27      0.12      0.17        24
         org       0.66      0.66      0.66      3656
         per       0.67      0.70      0.69      2782
         tim       0.84      0.80      0.82      2229

   micro avg       0.78      0.78      0.78     16782
   macro avg       0.55      0.52      0.53     16782
weighted avg       0.77      0.78      0.78     16782



##### Inference on New Text

In [ ]:
import torch
model.to("cuda" if torch.cuda.is_available() else "cpu")


def predict(text):
    tokens = tokenizer(text.split(), is_split_into_words=True, return_tensors="pt")
    tokens = {k: v.to(model.device) for k, v in tokens.items()}  # move to same device as model

    outputs = model(**tokens)
    predictions = outputs.logits.argmax(-1).squeeze().tolist()
    word_ids = tokens['input_ids'].cpu().squeeze().tolist()  # word_ids must be computed on CPU tokenizer

    word_map = tokenizer(text.split(), is_split_into_words=True).word_ids()
    final_predictions = []
    previous_word_idx = None

    for idx, word_idx in enumerate(word_map):
        if word_idx is None:
            continue
        if word_idx != previous_word_idx:
            label = id2label[predictions[idx]]
            final_predictions.append((text.split()[word_idx], label))
        previous_word_idx = word_idx

    return final_predictions

# Example 1
print(predict("Barack Obama visited Egypt in 2010."))

# Example 2
print(predict("Apple Inc. is based in California."))

[('Barack', np.str_('B-geo')), ('Obama', np.str_('B-geo')), ('visited', np.str_('I-gpe')), ('Egypt', np.str_('O')), ('in', np.str_('O')), ('2010.', np.str_('B-org'))]
[('Apple', np.str_('O')), ('Inc.', np.str_('O')), ('is', np.str_('O')), ('based', np.str_('O')), ('in', np.str_('O')), ('California.', np.str_('O'))]


#### DeBERTa

##### Training

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
import os
os.environ["WANDB_DISABLED"] = "true"

model_name = "microsoft/deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

data_collator = DataCollatorForTokenClassification(tokenizer)

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="./results/deberta",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=7,
    weight_decay=0.01,
    logging_steps=500,
    save_steps=1000,
    save_total_limit=1,
    fp16=True,
    optim="adamw_torch",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

trainer.train()

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Some weights of DebertaV2ForTokenClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

Step,Training Loss
500,0.915300
1000,0.739300
1500,0.599100
2000,0.537400
2500,0.479400
3000,0.444500
3500,0.414200
4000,0.381700
4500,0.369700
5000,0.331000


TrainOutput(global_step=37772, training_loss=0.22465470649422722, metrics={'train_runtime': 8657.5511, 'train_samples_per_second': 34.899, 'train_steps_per_second': 4.363, 'total_flos': 2.4058306465941384e+16, 'train_loss': 0.22465470649422722, 'epoch': 7.0})

##### Evaluation

In [ ]:
from seqeval.metrics import classification_report

preds = trainer.predict(val_dataset)
predictions = preds.predictions.argmax(-1)
label_ids = preds.label_ids

true_labels = []
pred_labels = []

for pred, label in zip(predictions, label_ids):
    true_seq = []
    pred_seq = []
    for p, l in zip(pred, label):
        if l != -100:
            true_seq.append(id2label[l])
            pred_seq.append(id2label[p])
    true_labels.append(true_seq)
    pred_labels.append(pred_seq)

print(classification_report(true_labels, pred_labels))

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


              precision    recall  f1-score   support

         art       0.00      0.00      0.00        92
         eve       0.38      0.12      0.18        42
         geo       0.82      0.88      0.85      6167
         gpe       0.93      0.90      0.92      1790
         nat       0.00      0.00      0.00        24
         org       0.69      0.66      0.68      3656
         per       0.70      0.72      0.71      2782
         tim       0.83      0.83      0.83      2229

   micro avg       0.79      0.79      0.79     16782
   macro avg       0.55      0.51      0.52     16782
weighted avg       0.78      0.79      0.79     16782



###### Inference on New Text

In [ ]:
import torch
model.to("cuda" if torch.cuda.is_available() else "cpu")


def predict(text):
    tokens = tokenizer(text.split(), is_split_into_words=True, return_tensors="pt")
    tokens = {k: v.to(model.device) for k, v in tokens.items()}  # move to same device as model

    outputs = model(**tokens)
    predictions = outputs.logits.argmax(-1).squeeze().tolist()
    word_ids = tokens['input_ids'].cpu().squeeze().tolist()  # word_ids must be computed on CPU tokenizer

    word_map = tokenizer(text.split(), is_split_into_words=True).word_ids()
    final_predictions = []
    previous_word_idx = None

    for idx, word_idx in enumerate(word_map):
        if word_idx is None:
            continue
        if word_idx != previous_word_idx:
            label = id2label[predictions[idx]]
            final_predictions.append((text.split()[word_idx], label))
        previous_word_idx = word_idx

    return final_predictions

# Example 1
print(predict("Barack Obama visited Egypt in 2010."))

# Example 2
print(predict("Apple Inc. is based in California."))

[('Barack', np.str_('O')), ('Obama', np.str_('O')), ('visited', np.str_('B-tim')), ('Egypt', np.str_('I-tim')), ('in', np.str_('I-tim')), ('2010.', np.str_('I-tim'))]
[('Apple', np.str_('B-tim')), ('Inc.', np.str_('O')), ('is', np.str_('B-tim')), ('based', np.str_('I-tim')), ('in', np.str_('I-tim')), ('California.', np.str_('O'))]


#### IndoBERT

##### Training

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
import os
os.environ["WANDB_DISABLED"] = "true"

model_name = "indobenchmark/indobert-base-p1"

tokenizer = AutoTokenizer.from_pretrained(model_name)

data_collator = DataCollatorForTokenClassification(tokenizer)

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="./results/indobert",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=7,
    weight_decay=0.01,
    logging_steps=500,
    save_steps=1000,
    save_total_limit=1,
    fp16=True,
    optim="adamw_torch",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

trainer.train()

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at indobenchmark/indobert-base-p1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

Step,Training Loss
500,0.554600
1000,0.331500
1500,0.269100
2000,0.249800
2500,0.228400
3000,0.210700
3500,0.206600
4000,0.192400
4500,0.195200
5000,0.173700


TrainOutput(global_step=37772, training_loss=0.09205070852816742, metrics={'train_runtime': 5561.41, 'train_samples_per_second': 54.328, 'train_steps_per_second': 6.792, 'total_flos': 2.405787207903425e+16, 'train_loss': 0.09205070852816742, 'epoch': 7.0})

##### Evaluation

In [ ]:
from seqeval.metrics import classification_report

preds = trainer.predict(val_dataset)
predictions = preds.predictions.argmax(-1)
label_ids = preds.label_ids

true_labels = []
pred_labels = []

for pred, label in zip(predictions, label_ids):
    true_seq = []
    pred_seq = []
    for p, l in zip(pred, label):
        if l != -100:
            true_seq.append(id2label[l])
            pred_seq.append(id2label[p])
    true_labels.append(true_seq)
    pred_labels.append(pred_seq)

print(classification_report(true_labels, pred_labels))

              precision    recall  f1-score   support

         art       0.10      0.08      0.09        92
         eve       0.28      0.26      0.27        42
         geo       0.85      0.88      0.87      6167
         gpe       0.94      0.93      0.94      1790
         nat       0.47      0.29      0.36        24
         org       0.72      0.71      0.71      3656
         per       0.75      0.78      0.76      2782
         tim       0.85      0.84      0.84      2229

   micro avg       0.81      0.82      0.81     16782
   macro avg       0.62      0.59      0.60     16782
weighted avg       0.81      0.82      0.81     16782



##### Inference on New Text

In [ ]:
import torch
model.to("cuda" if torch.cuda.is_available() else "cpu")


def predict(text):
    tokens = tokenizer(text.split(), is_split_into_words=True, return_tensors="pt")
    tokens = {k: v.to(model.device) for k, v in tokens.items()}  # move to same device as model

    outputs = model(**tokens)
    predictions = outputs.logits.argmax(-1).squeeze().tolist()
    word_ids = tokens['input_ids'].cpu().squeeze().tolist()  # word_ids must be computed on CPU tokenizer

    word_map = tokenizer(text.split(), is_split_into_words=True).word_ids()
    final_predictions = []
    previous_word_idx = None

    for idx, word_idx in enumerate(word_map):
        if word_idx is None:
            continue
        if word_idx != previous_word_idx:
            label = id2label[predictions[idx]]
            final_predictions.append((text.split()[word_idx], label))
        previous_word_idx = word_idx

    return final_predictions

# Example 1
print(predict("Barack Obama visited Egypt in 2010."))

# Example 2
print(predict("Apple Inc. is based in California."))

[('Barack', np.str_('O')), ('Obama', np.str_('O')), ('visited', np.str_('O')), ('Egypt', np.str_('O')), ('in', np.str_('O')), ('2010.', np.str_('O'))]
[('Apple', np.str_('O')), ('Inc.', np.str_('O')), ('is', np.str_('O')), ('based', np.str_('O')), ('in', np.str_('O')), ('California.', np.str_('O'))]


#### XLM-RoBERTa

##### Training

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
import os
os.environ["WANDB_DISABLED"] = "true"

model_name = "xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

data_collator = DataCollatorForTokenClassification(tokenizer)

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="./results/xlm-roberta",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=7,
    weight_decay=0.01,
    logging_steps=500,
    save_steps=1000,
    save_total_limit=1,
    fp16=True,
    optim="adamw_torch",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

trainer.train()

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
500,0.796600
1000,0.526600
1500,0.414900
2000,0.367100
2500,0.323500
3000,0.298100
3500,0.285100
4000,0.267000
4500,0.261800
5000,0.236300


TrainOutput(global_step=37772, training_loss=0.15814890877165103, metrics={'train_runtime': 11493.9607, 'train_samples_per_second': 26.287, 'train_steps_per_second': 3.286, 'total_flos': 2.405787207903425e+16, 'train_loss': 0.15814890877165103, 'epoch': 7.0})

##### Evaluation

In [ ]:
from seqeval.metrics import classification_report

preds = trainer.predict(val_dataset)
predictions = preds.predictions.argmax(-1)
label_ids = preds.label_ids

true_labels = []
pred_labels = []

for pred, label in zip(predictions, label_ids):
    true_seq = []
    pred_seq = []
    for p, l in zip(pred, label):
        if l != -100:
            true_seq.append(id2label[l])
            pred_seq.append(id2label[p])
    true_labels.append(true_seq)
    pred_labels.append(pred_seq)

print(classification_report(true_labels, pred_labels))

              precision    recall  f1-score   support

         art       0.00      0.00      0.00        92
         eve       0.48      0.29      0.36        42
         geo       0.84      0.89      0.87      6167
         gpe       0.95      0.92      0.93      1790
         nat       0.50      0.21      0.29        24
         org       0.72      0.70      0.71      3656
         per       0.73      0.75      0.74      2782
         tim       0.82      0.83      0.83      2229

   micro avg       0.80      0.81      0.81     16782
   macro avg       0.63      0.57      0.59     16782
weighted avg       0.80      0.81      0.81     16782



##### Inference on New Text

In [ ]:
import torch
model.to("cuda" if torch.cuda.is_available() else "cpu")


def predict(text):
    tokens = tokenizer(text.split(), is_split_into_words=True, return_tensors="pt")
    tokens = {k: v.to(model.device) for k, v in tokens.items()}  # move to same device as model

    outputs = model(**tokens)
    predictions = outputs.logits.argmax(-1).squeeze().tolist()
    word_ids = tokens['input_ids'].cpu().squeeze().tolist()  # word_ids must be computed on CPU tokenizer

    word_map = tokenizer(text.split(), is_split_into_words=True).word_ids()
    final_predictions = []
    previous_word_idx = None

    for idx, word_idx in enumerate(word_map):
        if word_idx is None:
            continue
        if word_idx != previous_word_idx:
            label = id2label[predictions[idx]]
            final_predictions.append((text.split()[word_idx], label))
        previous_word_idx = word_idx

    return final_predictions

# Example 1
print(predict("Barack Obama visited Egypt in 2010."))

# Example 2
print(predict("Apple Inc. is based in California."))

[('Barack', np.str_('O')), ('Obama', np.str_('O')), ('visited', np.str_('O')), ('Egypt', np.str_('B-org')), ('in', np.str_('O')), ('2010.', np.str_('B-geo'))]
[('Apple', np.str_('O')), ('Inc.', np.str_('O')), ('is', np.str_('O')), ('based', np.str_('O')), ('in', np.str_('O')), ('California.', np.str_('B-org'))]


##### BERT-cased

Training

In [ ]:
from transformers import BertForTokenClassification , Trainer , TrainingArguments ,DataCollatorForTokenClassification
import os

os.environ["WANDB_DISABLED"] = "true"

data_collator = DataCollatorForTokenClassification(tokenizer)

model=BertForTokenClassification.from_pretrained(
    'bert-base-cased',
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=7,
    weight_decay=0.01,
    logging_steps=500,     # log every 500 steps
    save_steps=1000,       # save every 1000 steps
    save_total_limit=1     # keep only 1 checkpoint
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator # Removed tokenizer=tokenizer
)

trainer.train()

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Step,Training Loss
500,0.310500
1000,0.177300
1500,0.158300
2000,0.152900
2500,0.146100
3000,0.136500
3500,0.138600
4000,0.137600
4500,0.136900
5000,0.126100


TrainOutput(global_step=37772, training_loss=0.06836527605550617, metrics={'train_runtime': 9585.5476, 'train_samples_per_second': 31.52, 'train_steps_per_second': 3.941, 'total_flos': 2.405787207903425e+16, 'train_loss': 0.06836527605550617, 'epoch': 7.0})

Evaluation

In [ ]:
from seqeval.metrics import classification_report

preds = trainer.predict(val_dataset)
predictions = preds.predictions.argmax(-1)
label_ids = preds.label_ids

true_labels = []
pred_labels = []

for pred, label in zip(predictions, label_ids):
    true_seq = []
    pred_seq = []
    for p, l in zip(pred, label):
        if l != -100:
            true_seq.append(id2label[l])
            pred_seq.append(id2label[p])
    true_labels.append(true_seq)
    pred_labels.append(pred_seq)

print(classification_report(true_labels, pred_labels))

              precision    recall  f1-score   support

         art       0.18      0.14      0.16        92
         eve       0.33      0.33      0.33        42
         geo       0.87      0.90      0.89      6167
         gpe       0.95      0.94      0.95      1790
         nat       0.33      0.29      0.31        24
         org       0.77      0.74      0.75      3656
         per       0.80      0.84      0.82      2782
         tim       0.87      0.87      0.87      2229

   micro avg       0.84      0.85      0.85     16782
   macro avg       0.64      0.63      0.63     16782
weighted avg       0.84      0.85      0.84     16782



Inference on New Text

In [ ]:
import torch
model.to("cuda" if torch.cuda.is_available() else "cpu")


def predict(text):
    tokens = tokenizer(text.split(), is_split_into_words=True, return_tensors="pt")
    tokens = {k: v.to(model.device) for k, v in tokens.items()}  # move to same device as model

    outputs = model(**tokens)
    predictions = outputs.logits.argmax(-1).squeeze().tolist()
    word_ids = tokens['input_ids'].cpu().squeeze().tolist()  # word_ids must be computed on CPU tokenizer

    word_map = tokenizer(text.split(), is_split_into_words=True).word_ids()
    final_predictions = []
    previous_word_idx = None

    for idx, word_idx in enumerate(word_map):
        if word_idx is None:
            continue
        if word_idx != previous_word_idx:
            label = id2label[predictions[idx]]
            final_predictions.append((text.split()[word_idx], label))
        previous_word_idx = word_idx

    return final_predictions

# Example 1
print(predict("Barack Obama visited Egypt in 2010."))

# Example 2
print(predict("Apple Inc. is based in California."))

[('Barack', np.str_('B-per')), ('Obama', np.str_('I-per')), ('visited', np.str_('O')), ('Egypt', np.str_('B-geo')), ('in', np.str_('O')), ('2010.', np.str_('B-tim'))]
[('Apple', np.str_('B-org')), ('Inc.', np.str_('I-org')), ('is', np.str_('O')), ('based', np.str_('O')), ('in', np.str_('O')), ('California.', np.str_('B-geo'))]


### 1. Teknik 1 : Shallow Machine Learning

##### Training

TF-IDF + SVM

In [ ]:
# TF-IDF + SVM (Token-level NER baseline with Progress Bars)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import numpy as np

# Flatten tokens and labels
X_tokens = [tok for sent in train_sentences for tok in sent]
y_tokens = [tag for seq in train_labels for tag in seq]

if val_sentences is None or val_labels is None:
    X_train_tok, X_val_tok, y_train_tok, y_val_tok = train_test_split(
        X_tokens, y_tokens, test_size=0.2, random_state=42, stratify=y_tokens
    )
else:
    X_train_tok = X_tokens
    y_train_tok = y_tokens
    X_val_tok = [tok for sent in val_sentences for tok in sent]
    y_val_tok = [tag for seq in val_labels for tag in seq]

# TF-IDF vectorizer
print("Vectorizing tokens")
vec = TfidfVectorizer(analyzer='word', ngram_range=(1,4), max_features=5000)
X_train_vec = vec.fit_transform(tqdm(X_train_tok, desc="Fitting TF-IDF (train)"))
X_val_vec = vec.transform(tqdm(X_val_tok, desc="Transforming TF-IDF (val)"))

# Train SVM (Linear Kernel)
print("Training SVM model")
svm = LinearSVC(random_state=42, dual=False, class_weight='balanced')

svm.fit(X_train_vec, y_train_tok)
print("Training completed")

batch_size = 2000
preds = []
for i in tqdm(range(0, X_val_vec.shape[0], batch_size), desc="Predicting"):
    preds.extend(svm.predict(X_val_vec[i:i+batch_size]))
pred = np.array(preds)




Vectorizing tokens


Fitting TF-IDF (train):   0%|          | 0/944040 [00:00<?, ?it/s]

Transforming TF-IDF (val):   0%|          | 0/104535 [00:00<?, ?it/s]

Training SVM model
Training completed


/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Predicting:   0%|          | 0/53 [00:00<?, ?it/s]

In [ ]:
import joblib

joblib.dump(svm, "svm_ner_model.pkl")
joblib.dump(vec, "svm_vectorizer.pkl")


['svm_vectorizer.pkl']

##### Evaluation

In [ ]:
print("TF-IDF + SVM token-level results:")
print(classification_report(y_val_tok, pred, digits=4))


TF-IDF + SVM token-level results:
              precision    recall  f1-score   support

       B-art     0.0263    0.1522    0.0449        46
       B-eve     0.0719    0.6286    0.1290        35
       B-geo     0.7233    0.6769    0.6993      3797
       B-gpe     0.9264    0.8781    0.9016      1592
       B-nat     0.1230    0.7500    0.2113        20
       B-org     0.4971    0.3723    0.4257      2055
       B-per     0.6750    0.5318    0.5949      1668
       B-tim     0.6995    0.7580    0.7276      2033
       I-art     0.0049    0.0750    0.0091        40
       I-eve     0.0395    0.5385    0.0736        39
       I-geo     0.4738    0.5835    0.5229       713
       I-gpe     0.1613    0.3125    0.2128        16
       I-nat     0.0137    0.8571    0.0270         7
       I-org     0.3617    0.3996    0.3798      1699
       I-per     0.6638    0.5060    0.5743      1658
       I-tim     0.2456    0.3846    0.2998       585
           O     0.9534    0.9387    0.9460    

##### Inference on New Text

In [ ]:
import joblib

svm_model = joblib.load("svm_ner_model.pkl")
vectorizer = joblib.load("svm_vectorizer.pkl")

def predict(text):
    tokens = text.split()
    X = vectorizer.transform(tokens)
    y_pred = svm_model.predict(X)
    y_pred = [str(y) for y in y_pred]

    return list(zip(tokens, y_pred))

print(predict("Barack Obama visited Egypt in 2010."))
print(predict("Apple Inc. is based in California."))


[('Barack', 'I-per'), ('Obama', 'I-per'), ('visited', 'O'), ('Egypt', 'B-geo'), ('in', 'O'), ('2010.', 'B-tim')]
[('Apple', 'O'), ('Inc.', 'I-org'), ('is', 'O'), ('based', 'O'), ('in', 'O'), ('California.', 'B-geo')]


### 2. Teknik 2 : Deep Learning

##### Training

Word2Vec + CNN

In [14]:
import numpy as np
from gensim.models import Word2Vec
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Input, Embedding, Conv1D, Dropout, TimeDistributed, Dense
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split

# Parameters
max_len = 200
embedding_dim = 300
num_words = 20000

# Fit Word2Vec on training sentences
w2v = Word2Vec(sentences=train_sentences , vector_size=embedding_dim, window=5, min_count=1, workers=2, sg=1, epochs=5)

# Tokenizer
texts = [' '.join(s) for s in train_sentences]
tokenizer = Tokenizer(num_words=num_words, oov_token='<OOV>')
tokenizer.fit_on_texts(texts)
word_index = tokenizer.word_index
vocab_size = min(num_words, len(word_index)) + 1

# Build embedding matrix
embedding_matrix = np.random.normal(scale=0.6, size=(vocab_size, embedding_dim)).astype('float32')
for w, i in word_index.items():
    if i >= vocab_size:
        continue
    if w in w2v.wv:
        embedding_matrix[i] = w2v.wv[w]

# Convert sentences to sequences and pad
sequences = tokenizer.texts_to_sequences(texts)
X = pad_sequences(sequences, maxlen=max_len, padding='post', truncating='post')

# Prepare tag to index mapping
all_tags = sorted(list({t for seq in train_labels for t in seq}))
tag2idx = {t: i+1 for i, t in enumerate(all_tags)}
idx2tag = {i: t for t, i in tag2idx.items()}

y = []
for seq in train_labels:
    yseq = [tag2idx[t] for t in seq]
    if len(yseq) < max_len:
        yseq = yseq + [0] * (max_len - len(yseq))
    else:
        yseq = yseq[:max_len]
    y.append(yseq)
y = np.array(y)

# Split
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Build simple CNN model
num_tags = max(tag2idx.values()) + 1
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len, weights=[embedding_matrix], trainable=True, mask_zero=True),
    Conv1D(filters=128, kernel_size=3, padding='same', activation='relu'),
    Dropout(0.2),
    Conv1D(filters=64, kernel_size=3, padding='same', activation='relu'),
    TimeDistributed(Dense(num_tags, activation='softmax'))
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.build(input_shape=(None, max_len))
model.summary()

# Prepare y for sparse_categorical_crossentropy
y_tr_in = np.expand_dims(y_tr, -1)
y_val_in = np.expand_dims(y_val, -1)

history = model.fit(X_tr, y_tr_in, validation_data=(X_val, y_val_in), epochs=7, batch_size=8)
loss, acc = model.evaluate(X_val, y_val_in, verbose=0)
print({'val_loss': float(loss), 'val_accuracy': float(acc)})


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:965: UserWarning: Layer 'conv1d' (of type Conv1D) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 200, 300)       │     6,000,300 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 200, 128)       │       115,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 200, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 200, 64)        │        24,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 200, 18)        │         1,170 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,141,438 (23.43 MB)

 Trainable params: 6,141,438 (23.43 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/7
4317/4317 ━━━━━━━━━━━━━━━━━━━━ 158s 23ms/step - accuracy: 0.9790 - loss: 0.0855 - val_accuracy: 0.9830 - val_loss: 0.0576
Epoch 2/7
4317/4317 ━━━━━━━━━━━━━━━━━━━━ 34s 8ms/step - accuracy: 0.9837 - loss: 0.0546 - val_accuracy: 0.9834 - val_loss: 0.0562
Epoch 3/7
4317/4317 ━━━━━━━━━━━━━━━━━━━━ 34s 8ms/step - accuracy: 0.9848 - loss: 0.0498 - val_accuracy: 0.9836 - val_loss: 0.0568
Epoch 4/7
4317/4317 ━━━━━━━━━━━━━━━━━━━━ 37s 8ms/step - accuracy: 0.9856 - loss: 0.0468 - val_accuracy: 0.9836 - val_loss: 0.0569
Epoch 5/7
4317/4317 ━━━━━━━━━━━━━━━━━━━━ 37s 9ms/step - accuracy: 0.9861 - loss: 0.0447 - val_accuracy: 0.9836 - val_loss: 0.0584
Epoch 6/7
4317/4317 ━━━━━━━━━━━━━━━━━━━━ 35s 8ms/step - accuracy: 0.9868 - loss: 0.0423 - val_accuracy: 0.9834 - val_loss: 0.0595
Epoch 7/7
4317/4317 ━━━━━━━━━━━━━━━━━━━━ 34s 8ms/step - accuracy: 0.9873 - loss: 0.0409 - val_accuracy: 0.9832 - val_loss: 0.0619
{'val_loss': 0.061930038034915924, 'val_accuracy': 0.9831835031509399}


##### Evaluation

In [15]:
def remove_padding(preds, labels, pad_token=0):
    unpadded_preds, unpadded_labels = [], []
    for p_seq, l_seq in zip(preds, labels):
        for p, l in zip(p_seq, l_seq):
            if l != pad_token:
                unpadded_preds.append(p)
                unpadded_labels.append(l)
    return np.array(unpadded_preds), np.array(unpadded_labels)

In [16]:

import numpy as np
from sklearn.metrics import classification_report
from tensorflow.keras.preprocessing.sequence import pad_sequences

y_true_flat, y_pred_flat = [], []

y_val_pred = model.predict(X_val, verbose=0)
y_val_pred = np.argmax(y_val_pred, axis=-1)

# Hapus padding
# Hapus padding
y_pred_flat, y_true_flat = remove_padding(y_val_pred, y_val, pad_token=0)

# Buang juga prediksi yang kebetulan 0
mask = y_pred_flat != 0
y_pred_flat = y_pred_flat[mask]
y_true_flat = y_true_flat[mask]

# Konversi ke label string
y_true_tags = [idx2tag[y] for y in y_true_flat]
y_pred_tags = [idx2tag[y] for y in y_pred_flat]


print("CNN token-level results:")
print(classification_report(y_true_tags, y_pred_tags, digits=4))


CNN token-level results:
              precision    recall  f1-score   support

       B-art     0.1786    0.1111    0.1370        90
       B-eve     0.2400    0.2308    0.2353        52
       B-geo     0.5561    0.4301    0.4851      6684
       B-gpe     0.7141    0.5195    0.6014      2876
       B-nat     0.6000    0.2045    0.3051        44
       B-org     0.5346    0.3485    0.4219      3616
       B-per     0.5834    0.5432    0.5626      3008
       B-tim     0.6270    0.5032    0.5583      3585
       I-art     0.1579    0.0508    0.0769        59
       I-eve     0.0952    0.0455    0.0615        44
       I-geo     0.4737    0.4003    0.4339      1329
       I-gpe     0.6364    0.3684    0.4667        38
       I-nat     0.0000    0.0000    0.0000        12
       I-org     0.5508    0.4311    0.4837      2992
       I-per     0.5952    0.6275    0.6109      3060
       I-tim     0.5861    0.2867    0.3851      1151
           O     0.9277    0.9629    0.9449    155812

 

##### Inference on New Text

In [ ]:
def predict_sentence(text, tokenizer, model, idx2tag, max_len=128):
    # Tokenize text
    tokens = text.split()
    seq = tokenizer.texts_to_sequences([' '.join(tokens)])
    seq_padded = pad_sequences(seq, maxlen=max_len, padding='post', truncating='post')

    # Predict
    pred = model.predict(seq_padded, verbose=0)
    pred_labels = np.argmax(pred, axis=-1)[0][:len(tokens)]

    # Map back to tags
    results = [(tok, idx2tag.get(label_id, 'O')) for tok, label_id in zip(tokens, pred_labels)]
    return results

print("\n Example 1:")
print(predict_sentence("Barack Obama visited Egypt in 2010.", tokenizer, model, idx2tag))

print("\n Example 2:")
print(predict_sentence("Apple Inc. is based in California.", tokenizer, model, idx2tag))

## Evaluation

In [ ]:
# from seqeval.metrics import classification_report

# preds = trainer.predict(val_dataset)
# predictions = preds.predictions.argmax(-1)
# label_ids = preds.label_ids

# true_labels = []
# pred_labels = []

# for pred, label in zip(predictions, label_ids):
#     true_seq = []
#     pred_seq = []
#     for p, l in zip(pred, label):
#         if l != -100:
#             true_seq.append(id2label[l])
#             pred_seq.append(id2label[p])
#     true_labels.append(true_seq)
#     pred_labels.append(pred_seq)

# print(classification_report(true_labels, pred_labels))

## Inference on New Text [From Template

In [ ]:
# import torch
# model.to("cuda" if torch.cuda.is_available() else "cpu")


# def predict(text):
#     tokens = tokenizer(text.split(), is_split_into_words=True, return_tensors="pt")
#     tokens = {k: v.to(model.device) for k, v in tokens.items()}  # move to same device as model

#     outputs = model(**tokens)
#     predictions = outputs.logits.argmax(-1).squeeze().tolist()
#     word_ids = tokens['input_ids'].cpu().squeeze().tolist()  # word_ids must be computed on CPU tokenizer

#     word_map = tokenizer(text.split(), is_split_into_words=True).word_ids()
#     final_predictions = []
#     previous_word_idx = None

#     for idx, word_idx in enumerate(word_map):
#         if word_idx is None:
#             continue
#         if word_idx != previous_word_idx:
#             label = id2label[predictions[idx]]
#             final_predictions.append((text.split()[word_idx], label))
#         previous_word_idx = word_idx

#     return final_predictions

# # Example 1
# print(predict("Barack Obama visited Egypt in 2010."))

# # Example 2
# print(predict("Apple Inc. is based in California."))

In [ ]:
# def predict(text):
#     tokens = tokenizer(text.split(), is_split_into_words=True, return_tensors="pt")
#     tokens = {k: v.to(model.device) for k, v in tokens.items()}  # move to same device as model

#     outputs = model(**tokens)
#     predictions = outputs.logits.argmax(-1).squeeze().tolist()
#     word_ids = tokens['input_ids'].cpu().squeeze().tolist()  # word_ids must be computed on CPU tokenizer

#     word_map = tokenizer(text.split(), is_split_into_words=True).word_ids()
#     final_predictions = []
#     previous_word_idx = None

#     for idx, word_idx in enumerate(word_map):
#         if word_idx is None:
#             continue
#         if word_idx != previous_word_idx:
#             label = id2label[predictions[idx]]
#             final_predictions.append((text.split()[word_idx], label))
#         previous_word_idx = word_idx

#     return final_predictions

In [ ]:
# # Example 1
# print(predict("Barack Obama visited Egypt in 2010."))

# # Example 2
# print(predict("Apple Inc. is based in California."))